Creación de Variables Temporales (temperatura por medio, viento por medio y máximo) por coordenada y  en base a los últimos 15 días del dataset mediante SQL. Objetivo: obtener una visión más clara de las tendencias recientes en la temperatura del mar y el viento, y verificar su correlación, lo que puede ser crucial para identificar eventos de El Niño, para que las empresas puedan anticipar pérdidas y gestionar sus riesgos operativos.

In [ ]:
import sqlite3
import pandas as pd


def calcular_variables_temperatura(db_path: str) -> pd.DataFrame:
    """Paso 1: Calcula el promedio de temperatura."""
    try:
        conexion = sqlite3.connect(db_path)
        query = """
            SELECT 
                a.complete_date AS complete_date,
                a.latitude,
                a.longitude,
                AVG(b.sst) AS sst_avg_fortnight
            FROM datos_el_nino a
            JOIN datos_el_nino b 
                ON a.latitude = b.latitude 
               AND a.longitude = b.longitude
               AND b.complete_date BETWEEN date(a.complete_date, '-14 days') AND a.complete_date
            GROUP BY a.complete_date, a.latitude, a.longitude;
        """
        print("[FUNCION 1] Calculando histórico de promedios de temperatura...")
        df_temp = pd.read_sql_query(query, conexion)
        return df_temp
    except sqlite3.Error as error:
        print(f"Error en calcular_variables_temperatura: {error}")
        raise
    finally:
        if "conexion" in locals():
            conexion.close()


In [ ]:
def calcular_variables_viento(db_path: str) -> pd.DataFrame:
    """Paso 2: Calcula el viento máximo y promedio de los últimos 15 días

    para cada fecha y coordenada histórica.
    """
    try:
        conexion = sqlite3.connect(db_path)
        query = """
            SELECT 
                a.complete_date AS complete_date,
                a.latitude,
                a.longitude,
                MAX(b.zonal_winds) AS wind_max_fortnight,
                AVG(b.zonal_winds) AS avg_zonal_wind_15d
            FROM datos_el_nino a
            JOIN datos_el_nino b 
                ON a.latitude = b.latitude 
               AND a.longitude = b.longitude
               AND b.complete_date BETWEEN date(a.complete_date, '-14 days') AND a.complete_date
            GROUP BY a.complete_date, a.latitude, a.longitude;
        """
        print("[FUNCION 2] Calculando histórico de estadísticas de viento...")
        df_viento = pd.read_sql_query(query, conexion)
        return df_viento
    except sqlite3.Error as error:
        print(f"Error en calcular_variables_viento: {error}")
        raise
    finally:
        if "conexion" in locals():
            conexion.close()

# Creamos la tabla para aplicar al análisis exploratorio de datos (EDA)

Aquí haremos un inner join, por coordenada (latitude, longitude) y fecha (complete_date), entre la tabla original y la tabla de variables temporales, para obtener un dataset enriquecido que contenga tanto las métricas originales como las nuevas variables calculadas. Esto nos permitirá realizar un análisis más completo y detectar patrones o tendencias que podrían ser indicativos de eventos de El Niño.

In [54]:
def crear_y_exportar_dataset_puro(
    db_path: str,
    df_temp: pd.DataFrame,
    df_viento: pd.DataFrame,
    output_csv_path: str,
) -> None:
    """Paso 3: Combina las variables físicas excluyendo por completo alert_anomaly,

    crea la nueva tabla en la base de datos y exporta el CSV limpio.
    """
    try:
        conexion = sqlite3.connect(db_path)

        # 1. Traer los datos base diarios (excluyendo alert_anomaly)
        query_base = """
            SELECT 
                complete_date AS complete_date,
                latitude,
                longitude,
                zonal_winds,
                meridional_winds,
                sst AS sst_today
            FROM datos_el_nino;
        """
        df_base = pd.read_sql_query(query_base, conexion)

        print("[PASO 3] Consolidando el dataset oceanográfico puro...")

        # 2. Combinar las tablas calculadas mediante intersección estricta
        df_final = pd.merge(
            df_base, df_temp, on=["complete_date", "latitude", "longitude"], how="inner"
        )
        df_final = pd.merge(
            df_final, df_viento, on=["complete_date", "latitude", "longitude"], how="inner"
        )

        # Definir el orden lógico de las columnas puramente físicas
        columnas_ordenadas = [
            "complete_date",
            "latitude",
            "longitude",
            "zonal_winds",
            "meridional_winds",
            "sst_today",
            "sst_avg_fortnight",
            "wind_max_fortnight",
            "avg_zonal_wind_15d",
        ]
        df_final = df_final[columnas_ordenadas]

        # 3. Crear y guardar los datos en la nueva tabla física de la BD con el nombre actualizado
        nombre_tabla = "table_atmospheric_oceanic_dataset"
        print(f"-> Guardando tabla '{nombre_tabla}' en la base de datos...")
        df_final.to_sql(
            name=nombre_tabla,
            con=conexion,
            if_exists="replace",
            index=False,
        )

        # 4. Exportar al nuevo archivo CSV físico
        print(f"-> Exportando datos limpios a '{output_csv_path}'...")
        df_final.to_csv(output_csv_path, index=False)
        print("¡Proceso finalizado! Dataset generado sin variables artificiales.")

    except Exception as error:
        print(f"Error en crear_y_exportar_dataset_puro: {error}")
        raise
    finally:
        if "conexion" in locals():
            conexion.close()

In [53]:
def ejecutar_pipeline_analisis_datos(db_path: str, csv_path: str) -> None:
    """Orquestador que coordina la generación del tablón oceanográfico."""
    print("=== INICIANDO CONSTRUCCIÓN DEL DATASET ATMOSFÉRICO-OCEÁNICO ===")

    datos_temperatura = calcular_variables_temperatura(db_path)
    datos_viento = calcular_variables_viento(db_path)
    crear_y_exportar_dataset_puro(db_path, datos_temperatura, datos_viento, csv_path)

    print("=== PROCESO COMPLETADO CON ÉXITO ===")


if __name__ == "__main__":
    # Definimos los nuevos nombres comerciales y académicos para tu proyecto
    ruta_db = "proyecto_final_elnino.db"
    ruta_salida_csv = "table_atmospheric_oceanic_dataset.csv"

    ejecutar_pipeline_analisis_datos(ruta_db, ruta_salida_csv)

=== INICIANDO CONSTRUCCIÓN DEL DATASET ATMOSFÉRICO-OCEÁNICO ===
[FUNCION 1] Calculando histórico de promedios de temperatura...
[FUNCION 2] Calculando histórico de estadísticas de viento...
[PASO 3] Consolidando el dataset oceanográfico puro...
-> Guardando tabla 'table_atmospheric_oceanic_dataset' en la base de datos...
-> Exportando datos limpios a 'table_atmospheric_oceanic_dataset.csv'...
¡Proceso finalizado! Dataset generado sin variables artificiales.
=== PROCESO COMPLETADO CON ÉXITO ===
